# DBSCAN

DBSCAN (Density-Based Spatial Clustering) groups points that are packed closely
together and labels sparse points as **noise**. Unlike
[k-means](kmeans.ipynb), you don't tell it how many clusters to find — it
discovers that from the density.

In `linfa`, DBSCAN is a *transformer*: it maps points directly to cluster
assignments, with no separate fit/predict step.

In [ ]:
:dep ndarray = { version = "0.15" }
:dep linfa = { version = "0.7" }
:dep linfa-clustering = { version = "0.7" }
use ndarray::array;
use linfa::traits::Transformer;
use linfa_clustering::Dbscan;

// Two dense blobs plus one isolated point (which DBSCAN should call noise).
let data = array![
    [1.0_f64, 1.0], [1.1, 0.9], [0.9, 1.1],
    [8.0, 8.0], [8.1, 7.9], [7.9, 8.1],
    [4.5, 4.5]
];

// min_points = 3 (min cluster size), tolerance = neighbourhood radius.
let clusters = Dbscan::params(3)
    .tolerance(0.5)
    .transform(&data)
    .expect("dbscan failed");
println!("assignments: {:?}", clusters);

Each entry is an `Option<usize>`: `Some(id)` for a cluster member, `None` for
noise. The isolated point at `(4.5, 4.5)` should be `None`. Let's plot noise in
black and each cluster in its own colour:

In [ ]:
:dep plotters = { version = "0.3", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
use plotters::prelude::*;

evcxr_figure((420, 360), |root| {
    root.fill(&WHITE)?;
    let mut chart = ChartBuilder::on(&root)
        .caption("DBSCAN (black = noise)", ("sans-serif", 18))
        .margin(10)
        .x_label_area_size(30)
        .y_label_area_size(30)
        .build_cartesian_2d(0f64..10f64, 0f64..10f64)?;
    chart.configure_mesh().draw()?;
    chart.draw_series((0..data.nrows()).map(|i| {
        let colour = match clusters[i] {
            Some(0) => RED,
            Some(_) => BLUE,
            None => BLACK,
        };
        Circle::new((data[[i, 0]], data[[i, 1]]), 5, colour.filled())
    }))?;
    Ok(())
})

Next: [decision trees](../04-trees/decision-trees.ipynb) — back to supervised
learning, with a model you can read as a set of if/else rules.